# Seam Diagnostics --- UV Duplicate-Vertex Artifact

Demonstrates the UV-seam duplicate-vertex artifact on an Einstar photogrammetry scan
and shows that a position-based merge eliminates it. Used to generate the
before/after figures for section 3.4 of the thesis.

**Pipeline:**
1. Load Subject 17
2. Pick the 5 landmarks ($N_z$, $I_z$, $C_z$, Lpa, Rpa) with the upstream cedalion picker
3. `normalize_axes` -> `isolate_head` -> `align_axes_from_landmarks` (CTF frame)
4. Render the isolated head with boundary-edge overlay -> **figure A (before)**
5. Collapse seam duplicates with a union-find merge
6. Render the merged head -> **figure B (after)**
7. Quantify the merge (vertex / face / component / boundary-edge counts)
8. Verify vertex colors survive averaging at seam positions

In [11]:
import logging
from collections import Counter

import numpy as np
import pyvista as pv
import trimesh
import xarray as xr
from scipy.spatial import KDTree
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

import cedalion
import cedalion.dataclasses as cdc
import cedalion.io
import cedalion.vis.blocks
from cedalion.geometry.photogrammetry.anonymization import (
    normalize_axes,
    isolate_head,
    align_axes_from_landmarks,
)

logging.getLogger('cedalion').setLevel(logging.WARNING)
pv.set_jupyter_backend('server')

# === CONFIG ===
SUBJECT_NUMBER = 17
SCANS_FOLDER = '/home/ma7/BA/PG_Subjects'

# True: run the picker. False: use cached landmarks (fill after first pick).
INTERACTIVE = True

# Position tolerance for seam merging (mm). Einstar seam duplicates are
# coincident within well under 0.01 mm.
MERGE_TOL_MM = 0.01

## 1. Load the Einstar scan

In [12]:
path = f'{SCANS_FOLDER}/Subject{SUBJECT_NUMBER}/Subject{SUBJECT_NUMBER}.obj'
surface = cedalion.io.read_einstar_obj(path)
print(f'Loaded: {surface.nvertices:,} vertices, {surface.nfaces:,} faces')

# In trimesh 4.6 the texture image lives on visual.material.image, not
# visual.image. Verify it's there; if missing, fall back to the sibling JPG.
import os
import trimesh as _tm
from PIL import Image as _PILImage

def _visual_image(visual):
    img = getattr(visual, 'image', None)
    if img is None:
        mat = getattr(visual, 'material', None)
        img = getattr(mat, 'image', None) if mat is not None else None
    return img

_img = _visual_image(surface.mesh.visual)
if _img is None:
    _jpg = path.replace('.obj', '.jpg')
    _uv = getattr(surface.mesh.visual, 'uv', None)
    assert os.path.exists(_jpg) and _uv is not None, (
        f'no texture: jpg_exists={os.path.exists(_jpg)}, has_uv={_uv is not None}'
    )
    surface.mesh.visual = _tm.visual.TextureVisuals(
        uv=_uv, image=_PILImage.open(_jpg).convert('RGBA'),
    )
    _img = _visual_image(surface.mesh.visual)
    assert _img is not None, 'attach failed'
    print(f'Attached texture from JPG: {_img.size}')
else:
    print(f'Texture already attached: {_img.size}')

Loaded: 1,284,667 vertices, 2,223,716 faces
Texture already attached: (4016, 3776)


## 2. Pick the 5 landmarks interactively

Uses cedalion's built-in picker (`cedalion.vis.blocks.plot_surface(..., pick_landmarks=True)`).
Right-click on the mesh to place a sphere; click again on the sphere to cycle
its label through `Nz -> Iz -> Cz -> Lpa -> Rpa`, or cycle past `Rpa` back to
`Nz` to remove it.

Close the window when all 5 are placed with correct labels.

In [13]:
if INTERACTIVE:
    pvplt = pv.Plotter()
    get_landmarks = cedalion.vis.blocks.plot_surface(
        pvplt, surface, opacity=1.0, pick_landmarks=True
    )
    pvplt.show()

Widget(value='<iframe src="http://localhost:37333/index.html?ui=P_0x7f00dc983690_4&reconnect=auto" class="pyvi…

## 3. Retrieve picked points and wrap into a `LabeledPointCloud`

In [14]:
if INTERACTIVE:
    landmark_coordinates, landmark_labels = get_landmarks()
    landmark_coordinates = np.asarray(landmark_coordinates)
else:
    # Cached fallback: paste coordinates from a previous interactive run here.
    landmark_labels = ['Nz', 'Iz', 'Cz', 'Lpa', 'Rpa']
    landmark_coordinates = np.asarray([
        [0.0, 0.0, 0.0],   # Nz  -- replace
        [0.0, 0.0, 0.0],   # Iz  -- replace
        [0.0, 0.0, 0.0],   # Cz  -- replace
        [0.0, 0.0, 0.0],   # Lpa -- replace
        [0.0, 0.0, 0.0],   # Rpa -- replace
    ])

assert len(set(landmark_labels)) == 5, 'Need exactly 5 distinct landmarks'
assert set(landmark_labels) == {'Nz', 'Iz', 'Cz', 'Lpa', 'Rpa'}, (
    f'Unexpected labels: {landmark_labels}'
)

landmarks = xr.DataArray(
    np.vstack(landmark_coordinates),
    dims=['label', 'digitized'],
    coords={
        'label': ('label', list(landmark_labels)),
        'type': ('label', [cdc.PointType.LANDMARK] * 5),
        'group': ('label', ['L'] * 5),
    },
).pint.quantify('mm')

display(landmarks)

Magnitude,[[183.263908273029 116.4407663790885 433.14976121066945] [55.810028864582236 216.87724678041928 460.20226543224396] [75.44543707953403 70.2217429163498 571.2643026729717] [70.90973965837875 -12.700521103862059 479.10120764275973] [-5.238211533933473 60.433665290097025 422.0801594510126]]
Units,millimeter


## 4. Normalize, isolate head, align to CTF

Same preprocessing as notebook 48. The output `surface_h` carries `crs='ctf'`.
Crucially, `isolate_head` uses only a *transient* position-based adjacency
graph to compute connected components, so the mesh it returns **still contains**
the seam-duplicate vertices --- which is the point of this diagnostic.

In [15]:
lm_raw = landmarks.pint.dequantify().values
idx = {lbl: i for i, lbl in enumerate(landmarks['label'].values)}
Nz_raw = lm_raw[idx['Nz']]

surface_n, Nz_n, R = normalize_axes(surface, Nz_raw)
landmarks_n = landmarks.pint.dequantify().copy(data=lm_raw @ R.T).pint.quantify()

surface_n, _ = isolate_head(surface_n, Nz_n)
print(f'After isolate_head: {surface_n.nvertices:,} vertices')

surface_h, landmarks_n, M_ctf = align_axes_from_landmarks(surface_n, landmarks_n)
print(f'After align_axes_from_landmarks: crs={surface_h.crs!r}, '
      f'{surface_h.nvertices:,} vertices')

After isolate_head: 678,081 vertices
After align_axes_from_landmarks: crs='ctf', 678,081 vertices


## 5. Visualize seam lines on the isolated head (before merge)

Boundary edges are edges that belong to exactly one face instead of the usual
two. On a watertight surface there should be essentially none. On a raw Einstar
scan, each UV patch boundary becomes a chain of boundary edges because the two
sides of the patch live on different, spatially coincident vertices.

In [16]:
def boundary_edges_of(mesh):
    """Return (boundary_edge_array, total_edge_count) for a trimesh."""
    edge_to_face_count = Counter()
    for face in mesh.faces:
        edge_to_face_count[tuple(sorted((int(face[0]), int(face[1]))))] += 1
        edge_to_face_count[tuple(sorted((int(face[1]), int(face[2]))))] += 1
        edge_to_face_count[tuple(sorted((int(face[0]), int(face[2]))))] += 1
    boundary = np.array(
        [e for e, c in edge_to_face_count.items() if c == 1],
        dtype=np.int64,
    )
    return boundary, len(edge_to_face_count)


def _boundary_polydata(mesh, boundary_edges):
    """Build a pyvista PolyData of line segments for the given boundary edges."""
    verts = mesh.vertices[boundary_edges.ravel()]
    n = len(boundary_edges)
    lines = np.column_stack([
        np.full(n, 2),
        np.arange(0, 2 * n, 2),
        np.arange(1, 2 * n, 2),
    ]).ravel()
    return pv.PolyData(verts, lines=lines)


boundary_before, total_edges_before = boundary_edges_of(surface_h.mesh)
print(f'Boundary edges: {len(boundary_before):,} / {total_edges_before:,} '
      f'({100 * len(boundary_before) / total_edges_before:.1f} %)')

boundary_poly_before = _boundary_polydata(surface_h.mesh, boundary_before)

pvplt = pv.Plotter()
cedalion.vis.blocks.plot_surface(pvplt, surface_h, opacity=1.0)
pvplt.add_mesh(boundary_poly_before, color='red', line_width=2, opacity=1.0)
pvplt.add_text(
    f'Before merge - seam lines: {len(boundary_before):,}',
    position='upper_left', font_size=14,
)
pvplt.show()

Boundary edges: 169,543 / 1,842,329 (9.2 %)


Widget(value='<iframe src="http://localhost:37333/index.html?ui=P_0x7f00379f9350_5&reconnect=auto" class="pyvi…

## 6. Collapse seam duplicates (persistent merge)

The same mechanism `isolate_head` applies transiently for connected-component
analysis, but here the mesh is actually rewritten: every group of spatially
coincident vertices (within `MERGE_TOL_MM`) is replaced by a single vertex
carrying the group-averaged position and color.

In [17]:
def merge_seam_vertices(mesh, tol=MERGE_TOL_MM):
    """Merge spatially coincident vertices via union-find on KDTree pairs.

    Returns (merged_mesh, old_to_new, new_to_old).
    """
    verts = mesh.vertices
    n = len(verts)

    tree = KDTree(verts)
    pairs = tree.query_pairs(r=tol)

    parent = np.arange(n)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    for a, b in pairs:
        union(a, b)

    for i in range(n):
        parent[i] = find(i)

    unique_roots, inverse = np.unique(parent, return_inverse=True)
    old_to_new = inverse
    n_new = len(unique_roots)

    new_to_old = [[] for _ in range(n_new)]
    for old_idx in range(n):
        new_to_old[old_to_new[old_idx]].append(old_idx)

    new_verts = np.zeros((n_new, 3))
    for new_idx in range(n_new):
        old_indices = new_to_old[new_idx]
        new_verts[new_idx] = verts[old_indices].mean(axis=0)

    new_faces = old_to_new[mesh.faces]
    valid = (
        (new_faces[:, 0] != new_faces[:, 1])
        & (new_faces[:, 1] != new_faces[:, 2])
        & (new_faces[:, 0] != new_faces[:, 2])
    )
    new_faces = new_faces[valid]

    # Preserve per-vertex colors if present. ColorVisuals exposes
    # ``vertex_colors`` directly; TextureVisuals does not, so we skip
    # color averaging there (the texture image stays attached to the
    # original mesh and can be re-sampled downstream if needed).
    old_colors = getattr(mesh.visual, 'vertex_colors', None)
    merged_mesh = trimesh.Trimesh(
        vertices=new_verts, faces=new_faces, process=False,
    )
    if old_colors is not None and len(old_colors) == n:
        old_colors = np.asarray(old_colors, dtype=np.float64)
        new_colors = np.zeros((n_new, old_colors.shape[1]), dtype=np.float64)
        for new_idx in range(n_new):
            new_colors[new_idx] = old_colors[new_to_old[new_idx]].mean(axis=0)
        merged_mesh.visual.vertex_colors = new_colors.astype(np.uint8)

    return merged_mesh, old_to_new, new_to_old


merged_mesh, old_to_new, new_to_old = merge_seam_vertices(surface_h.mesh)
surface_merged = cdc.TrimeshSurface(
    mesh=merged_mesh, crs=surface_h.crs, units=surface_h.units,
)

print(f'Vertices: {len(surface_h.mesh.vertices):,} -> {len(merged_mesh.vertices):,}  '
      f'({len(surface_h.mesh.vertices) - len(merged_mesh.vertices):,} removed)')
print(f'Faces:    {len(surface_h.mesh.faces):,} -> {len(merged_mesh.faces):,}  '
      f'({len(surface_h.mesh.faces) - len(merged_mesh.faces):,} degenerate removed)')

Vertices: 678,081 -> 586,433  (91,648 removed)
Faces:    1,171,705 -> 1,171,681  (24 degenerate removed)


## 7. Visualize merged head (seams gone)

Side-by-side: same head before (left, seam lines in red) and after (right,
no seam lines). Linked camera views so rotation applies to both panels.

In [18]:
boundary_after, _ = boundary_edges_of(merged_mesh)
print(f'Boundary edges after merge: {len(boundary_after):,}')

pvplt = pv.Plotter(shape=(1, 2))

pvplt.subplot(0, 0)
cedalion.vis.blocks.plot_surface(pvplt, surface_h, opacity=1.0)
pvplt.add_mesh(boundary_poly_before, color='red', line_width=2, opacity=0.8)
pvplt.add_text(
    f'Before - seams: {len(boundary_before):,}',
    position='upper_left', font_size=12,
)

pvplt.subplot(0, 1)
cedalion.vis.blocks.plot_surface(pvplt, surface_merged, opacity=1.0)
if len(boundary_after) > 0:
    boundary_poly_after = _boundary_polydata(merged_mesh, boundary_after)
    pvplt.add_mesh(boundary_poly_after, color='red', line_width=2, opacity=0.8)
pvplt.add_text(
    f'After merge - seams: {len(boundary_after):,}',
    position='upper_left', font_size=12,
)

pvplt.link_views()
pvplt.show()

Boundary edges after merge: 1,269


Widget(value='<iframe src="http://localhost:37333/index.html?ui=P_0x7eff638cff10_6&reconnect=auto" class="pyvi…

## 8. Quantify the merge

In [ ]:
def count_components(m):
    """Count connected components on the raw (non-position-merged) adjacency."""
    n = len(m.vertices)
    edges = m.edges_unique
    rows = np.concatenate([edges[:, 0], edges[:, 1]])
    cols = np.concatenate([edges[:, 1], edges[:, 0]])
    data = np.ones(len(rows))
    A = csr_matrix((data, (rows, cols)), shape=(n, n))
    n_comp, _ = connected_components(A, directed=False)
    return n_comp


n_comp_before = count_components(surface_h.mesh)
n_comp_after = count_components(merged_mesh)

print('=== Merge statistics ===')
print(f'Vertices:        {len(surface_h.mesh.vertices):>10,} -> {len(merged_mesh.vertices):>10,}  '
      f'({len(surface_h.mesh.vertices) - len(merged_mesh.vertices):,} removed)')
print(f'Faces:           {len(surface_h.mesh.faces):>10,} -> {len(merged_mesh.faces):>10,}  '
      f'({len(surface_h.mesh.faces) - len(merged_mesh.faces):,} degenerate removed)')
print(f'Components:      {n_comp_before:>10,} -> {n_comp_after:>10,}')
print(f'Boundary edges:  {len(boundary_before):>10,} -> {len(boundary_after):>10,}')
print()

areas = merged_mesh.area_faces
degenerate = int((areas < 1e-10).sum())
print(f'Degenerate faces after merge (area < 1e-10): {degenerate}')
print(f'is_watertight: {merged_mesh.is_watertight}')
print(f'Surface area:  {surface_h.mesh.area:.1f} mm^2 (before) vs '
      f'{merged_mesh.area:.1f} mm^2 (after)')

## 9. Verify vertex colors survive averaging

Seam duplicates usually share the same or near-identical color, so averaging
at each merge group should leave the rendered appearance unchanged. The left
panel uses the untouched textured mesh; the right panel uses the merged mesh
with per-vertex colors averaged from the duplicate groups. If both look the
same, the merge preserves appearance.

In [ ]:
pvplt = pv.Plotter(shape=(1, 2))

pvplt.subplot(0, 0)
cedalion.vis.blocks.plot_surface(pvplt, surface_h, opacity=1.0)
pvplt.add_text('Before merge', position='upper_left', font_size=12)

pvplt.subplot(0, 1)
cedalion.vis.blocks.plot_surface(pvplt, surface_merged, opacity=1.0)
pvplt.add_text('After merge', position='upper_left', font_size=12)

pvplt.link_views()
pvplt.show()

orig_colors = getattr(surface_h.mesh.visual, 'vertex_colors', None)
if orig_colors is not None and len(orig_colors) == len(surface_h.mesh.vertices):
    orig_colors = np.asarray(orig_colors)[:, :3].astype(float)
    max_diffs = []
    for new_idx in range(len(merged_mesh.vertices)):
        old_indices = new_to_old[new_idx]
        if len(old_indices) > 1:
            old_cols = orig_colors[old_indices]
            spread = old_cols.max(axis=0) - old_cols.min(axis=0)
            max_diffs.append(spread.max())

    if max_diffs:
        max_diffs = np.array(max_diffs)
        print('Color spread at merged vertex groups (0-255 scale):')
        print(f'  Max spread:    {max_diffs.max():.1f}')
        print(f'  Mean spread:   {max_diffs.mean():.1f}')
        print(f'  Median spread: {np.median(max_diffs):.1f}')
        print(f'  Groups with spread > 10: {int((max_diffs > 10).sum())} / {len(max_diffs)}')
    else:
        print('No merged vertex groups with >1 member found.')
else:
    print('Textured mesh (no per-vertex color array) - visual comparison above is the proxy.')